<a href="https://colab.research.google.com/github/ArifahZhafirah/PEMBELAJARAN-MESIN/blob/main/JS09/Tugas-LAB02/PEMMESTUGAS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tugas Lab 2

In [1]:
# Install library
!pip install annoy
!pip install faiss-cpu
!pip install hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 5.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520036 sha256=333bffca4068a87f2e1026c25c23e3f598c8ef3583af55f723da228487554463
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 55.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=0492b5ee03c4f58c52fa3d050165506dbbaaa4416a19ee1389c914f83a4ab439
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705

In [13]:
import os
import pandas as pd
import numpy as np
import time
import faiss
import kagglehub
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [14]:
# Download dataset
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Path to dataset files: /kaggle/input/spotify-songs-with-attributes-and-lyrics


In [15]:
# Load dataset
df = pd.read_csv(os.path.join(path, 'songs_with_attributes_and_lyrics.csv'))
print(df.columns)  # cek nama kolom

Index(['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics'],
      dtype='object')


In [16]:
# Pilih fitur numerik
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
df = df.dropna(subset=features).reset_index(drop=True)  # buang baris kosong
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

k = 10  # jumlah nearest neighbors
dim = X_scaled.shape[1]
n_data = X_scaled.shape[0]

# Query point (pilih 1 track)
query_idx = 0
query = X_scaled[query_idx:query_idx+1]
print("Jumlah data:", n_data)
print("Track query:")
print(df.iloc[query_idx])

Jumlah data: 955320
Track query:
id                                             0Prct5TDjAnEgIqbxcldY9
name                                                                !
album_name                                                 UNDEN!ABLE
artists                                                      HELLYEAH
danceability                                                    0.415
energy                                                          0.605
key                                                                 7
loudness                                                      -11.157
mode                                                                1
speechiness                                                    0.0575
acousticness                                                  0.00116
instrumentalness                                                0.838
liveness                                                        0.471
valence                                                  

In [17]:
# Exact NN
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)

start = time.time()
distances_exact, idx_exact = nn.kneighbors(query)
time_exact = time.time() - start

print("=== Exact NN ===")
print("Indices:", idx_exact)
print("Distances:", distances_exact)
print("Waktu:", time_exact, "s")

=== Exact NN ===
Indices: [[     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]]
Distances: [[0.         0.88665658 0.90944272 0.93600774 0.93759984 0.94650489
  0.97633171 0.99541754 1.02076137 1.04078293]]
Waktu: 0.11251711845397949 s


In [18]:
# ANNOY (10 trees)
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X_scaled[i])
ann_index.build(10)  # 10 trees
build_annoy = time.time() - start

start = time.time()
idx_annoy, dist_annoy = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
time_annoy = time.time() - start

print("=== Annoy ===")
print("Indices:", idx_annoy)
print("Distances:", dist_annoy)
print("Build time:", build_annoy, "s")
print("Waktu:", time_annoy, "s")

=== Annoy ===
Indices: [0, 764272, 749223, 625227, 833164, 676733, 523698, 85956, 888081, 763625]
Distances: [0.0, 0.9094427227973938, 0.9375998377799988, 0.9763317108154297, 0.9954175353050232, 1.020761489868164, 1.0407829284667969, 1.045661211013794, 1.0485272407531738, 1.0519623756408691]
Build time: 24.65395212173462 s
Waktu: 0.0008242130279541016 s


In [19]:
# FAISS (IVF Flat)
nlist = 100   # n-cluster (inverted list)
quantizer = faiss.IndexFlatL2(dim)
faiss_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)

start = time.time()
faiss_index.train(X_scaled)
faiss_index.add(X_scaled)
build_faiss = time.time() - start

faiss_index.nprobe = 10

start = time.time()
dist_faiss, idx_faiss = faiss_index.search(query, k)
time_faiss = time.time() - start

print("=== FAISS (IVF Flat) ===")
print("Indices:", idx_faiss)
print("Distances:", dist_faiss)
print("Build time:", build_faiss, "s")
print("Waktu:", time_faiss, "s")

=== FAISS (IVF Flat) ===
Indices: [[     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]]
Distances: [[0.         0.7861599  0.82708603 0.87611043 0.87909347 0.89587146
  0.9532236  0.99085605 1.0419539  1.0832291 ]]
Build time: 0.5124454498291016 s
Waktu: 0.001733541488647461 s


In [20]:
# HNSW
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
build_hnsw = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
idx_hnsw, dist_hnsw = hnsw_index.knn_query(query, k=k)
time_hnsw = time.time() - start

print("=== HNSW ===")
print("Indices:", idx_hnsw)
print("Distances:", dist_hnsw)
print("Build time:", build_hnsw, "s")
print("Waktu:", time_hnsw, "s")

=== HNSW ===
Indices: [[     0 394553 764272 837727 749223  61511 625227 833164 676733 523698]]
Distances: [[0.         0.7861599  0.82708603 0.8761104  0.87909347 0.8958715
  0.9532235  0.9908561  1.0419539  1.0832292 ]]
Build time: 230.17228937149048 s
Waktu: 0.0009393692016601562 s


In [21]:
# Fungsi recall (sama seperti Lab 5)
def recall_at_k(I_pred, I_gt, k):
    correct = 0
    for i in range(len(I_pred)):
        correct += len(set(I_pred[i][:k]) & set(I_gt[i][:k]))
    return correct / (len(I_pred) * k)

recall_annoy = recall_at_k([idx_annoy], idx_exact, k)
recall_faiss = recall_at_k(idx_faiss, idx_exact, k)
recall_hnsw = recall_at_k(idx_hnsw, idx_exact, k)

# Tabel perbandingan
df_banding = pd.DataFrame({
    'Metode': ['Exact NN', 'ANNOY', 'FAISS', 'HNSW'],
    'Build time (s)': [0, build_annoy, build_faiss, build_hnsw],
    'Query time (ms)': [time_exact*1000, time_annoy*1000, time_faiss*1000, time_hnsw*1000],
    'Recall@10': [1.0, recall_annoy, recall_faiss, recall_hnsw]
})
df_banding

,Metode,Build time (s),Query time (ms),Recall@10
0,Exact NN,0.000000,112.517118,1.0
1,ANNOY,24.653952,0.824213,0.7
2,FAISS,0.512445,1.733541,1.0
3,HNSW,230.172289,0.939369,1.0


In [22]:
# Track terdekat hasil tiap metode (baris pertama = track query itu sendiri)
print("=== Exact NN ===")
print(df.iloc[idx_exact[0]])
print("\n=== ANNOY ===")
print(df.iloc[idx_annoy])
print("\n=== FAISS ===")
print(df.iloc[idx_faiss[0]])
print("\n=== HNSW ===")
print(df.iloc[idx_hnsw[0]])

=== Exact NN ===
                            id                                     name  \
0       0Prct5TDjAnEgIqbxcldY9                                        !   
394553  2vy246tF9zAM26wZVIr8bx            Intro (Who The Fuck Are You')   
764272  4ikbMYjg2Yn2zjB4TeiX8P                              Testimonium   
837727  1E1M2eJ2o9bFRck0ZHl7Iv  Ultimate Satisfaction - The Field Remix   
749223  2KvIO4ew1TGnz1VfTJk49g                                 Sæglópur   
61511   15jU0mzRZHQAI6lhO8cnCw                         Autumn Interlude   
625227  3jtaAUbFSzfoKv6bVVo563                                  Púrpura   
833164  0pcLVqykiOhWTSOcmYJKIb                         Turn from Heaven   
676733  1S8jdYiBiX2b7awbS0LuRo                       See You When I Die   
523698  3SNHoYD7tyDI1nAl5RyWGW                                   Mrityu   

                                  album_name               artists  \
0                                 UNDEN!ABLE              HELLYEAH   
394553           